In [1]:
!pip install weaviate langchain-weaviate

  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-win_amd64.whl.metadata (7.4 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/619.5 kB ? eta -:--:--
   ---------------------------------------- 619.5/619.5 kB 7.6 MB/s  0:00:00
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
Using cached pydantic_core-2.41.5-cp312-cp312-win_amd64.whl (2.0 MB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)

   ---- ----------------------------------- 1/9 [validators]
   ---- ----------------------------------- 1/9 [validators]
   ---- ----------------------------------- 1/9 [validators]
   ---- ----------------------------------- 1/9 [validators]
   ---- ----------------------------------- 1/9 [validators]
   ---- ----------------------------------- 1/9 [validators]
   ---- ----------------------------------- 1/9 [validators]
   ---- -----------

  You can safely remove it manually.


In [2]:
import os
os.environ["WEAVIATE_URL"] = "xyz"
os.environ["WEAVIATE_API_KEY"] = "xyz"

In [6]:
import weaviate
from weaviate.classes.init import Auth

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
)

In [7]:
print(client.is_ready())

True


In [8]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("weaviatedata.txt")
documents = loader.load()

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


In [11]:
documents

[Document(metadata={'source': 'weaviatedata.txt'}, page_content='Artificial Intelligence (AI) is a field of computer science focused on building systems that can perform tasks requiring human intelligence. These tasks include learning, reasoning, problem-solving, and understanding language.\n\nMachine Learning (ML) is a subset of AI that allows systems to learn from data instead of being explicitly programmed. Common types include supervised learning, unsupervised learning, and reinforcement learning.\n\nDeep Learning is a subset of machine learning that uses neural networks with many layers. It is widely used in image recognition, speech processing, and natural language processing.\n\nNatural Language Processing (NLP) enables machines to understand and generate human language. Applications include chatbots, translation systems, and sentiment analysis.\n\nTransformers are a type of neural network architecture that rely on self-attention mechanisms. They are the foundation of modern lar

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

len(docs)

13

In [13]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\Piyush\AppData\Local\Temp\ipykernel_20456\2985419238.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
sample_vec = embedding.embed_query("test")
dim = len(sample_vec)

print("Embedding Dimension:", dim)

Embedding Dimension: 384


In [16]:
class_name = "LangChainDocs"

if not client.collections.exists(class_name):
    client.collections.create(
        name=class_name,
        vectorizer_config=None  #we provide embeddings
    )

In [17]:
from langchain_weaviate import WeaviateVectorStore

vectordb = WeaviateVectorStore(
    client=client,
    index_name=class_name,
    text_key="text",
    embedding=embedding
)

In [18]:
vectordb.add_documents(docs)

['29f58eed-15dd-4f6c-a10f-46fbe76f900c',
 '778a6bab-d345-4578-8817-74bfbc9d27a3',
 'db0e9283-d49b-4936-a182-6f766ca00856',
 'bc4b2ea6-f1e4-4865-b2bd-e23d049d429b',
 '4bbaca17-0ec8-4258-83fb-d43a5c282bf7',
 'b463d98f-b021-4f2a-9696-fc2c76ce2e6a',
 '8076c842-2916-4705-a9a1-f80425fad116',
 'df4d2ee1-19ad-4b92-9e3a-2ff2f3fbdf55',
 'd42e03fc-2a26-47c6-bab5-c881dee03ab3',
 'fa8f6360-381f-4c10-ac7d-1a9d5910c846',
 '69486b7f-7e22-452f-9fe3-5b46ef85c3cd',
 '89b5ee8b-4a95-45b9-9ea8-46aba7c3ab42',
 '303e7105-7f0c-4254-b65b-280e93ba3d79']

In [19]:
results = vectordb.similarity_search("What is RAG?", k=2)

for r in results:
    print(r.page_content)
    print("-" * 50)

RAG (Retrieval-Augmented Generation) is a technique where relevant documents are retrieved and passed to a language model to generate better answers.

LangChain is a framework for building applications with LLMs. It provides tools for chaining models, retrieval, and memory.
--------------------------------------------------
Pinecone is a managed vector database used for storing and searching embeddings efficiently.

Chroma is a lightweight vector database often used for local development and experimentation.

Weaviate is an open-source vector database that supports hybrid search combining keyword and vector search.
--------------------------------------------------


In [21]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=100,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
C:\Users\Piyush\AppData\Local\Temp\ipykernel_20456\2196471821.py:11: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [22]:
from langchain_classic.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectordb.as_retriever(search_kwargs={"k": 3})
)

In [23]:
qa.invoke("What is RAG?")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'query': 'What is RAG?',
 'result': "Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\nRAG (Retrieval-Augmented Generation) is a technique where relevant documents are retrieved and passed to a language model to generate better answers.\n\nLangChain is a framework for building applications with LLMs. It provides tools for chaining models, retrieval, and memory.\n\nPinecone is a managed vector database used for storing and searching embeddings efficiently.\n\nChroma is a lightweight vector database often used for local development and experimentation.\n\nWeaviate is an open-source vector database that supports hybrid search combining keyword and vector search.\n\nRegularization techniques like L1 and L2 help prevent overfitting by penalizing large weights.\n\nAPIs (Application Programming Interfaces) allow different software systems to communicate with each other.\n\nQues